# PointNet++ 직접 구현 — ModelNet40 3D 분류

**논문**: PointNet++: Deep Hierarchical Feature Learning on Point Sets in a Metric Space  
**저자**: Charles R. Qi, Li Yi, Hao Su, Leonidas J. Guibas (Stanford, 2017)  
**목적**: 핵심 모듈(FPS, Ball Query, Set Abstraction)을 직접 구현하고 ModelNet40으로 평가


In [ ]:
# Cell 1 — 한글 폰트 설정 + 전체 임포트
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import os
import sys
import time
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

# open3d는 시각화 보조용 (GUI 없이 matplotlib으로 대체)
try:
    import open3d as o3d
    OPEN3D_AVAILABLE = True
    print(f"open3d 버전: {o3d.__version__}")
except ImportError:
    OPEN3D_AVAILABLE = False
    print("open3d 없음 — matplotlib 3D scatter로 대체")

# 환경 확인
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n사용 디바이스: {DEVICE}")

# 재현성 고정
torch.manual_seed(42)
np.random.seed(42)

# 데이터 저장 경로
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

## Cell 2 — 논문 핵심 개념: PointNet vs PointNet++

### PointNet (2017, Qi et al.) — 한계

PointNet은 포인트클라우드를 처음으로 딥러닝으로 처리한 혁신적인 논문이다.  
하지만 핵심 한계가 있다: **로컬 구조(local structure)를 완전히 무시**한다.

```
PointNet 처리 방식:
각 포인트 → 독립적으로 MLP 처리 → Max Pooling → 전체 특징 하나
```

즉, 이웃 포인트들 사이의 관계(엣지, 곡률, 표면 법선 등)를 전혀 학습하지 못한다.  
사람 얼굴의 코와 귀가 얼마나 가까운지, 어떤 방향으로 배치되어 있는지 알 수 없다.

---

### PointNet++ (NeurIPS 2017) — 핵심 아이디어: Hierarchical Learning

PointNet++는 이 문제를 **계층적(hierarchical) 구조**로 해결한다.  
CNN이 이미지에서 엣지 → 텍스처 → 부분 → 전체 순서로 학습하는 것처럼,  
포인트클라우드에서도 **작은 지역 → 중간 지역 → 전체** 순서로 특징을 추출한다.

```
PointNet++ 처리 방식:

원본 포인트 (N=2048)
    ↓  [Set Abstraction Layer 1]
    FPS → 512개 중심점 선택
    Ball Query → 각 중심점 반경 0.2 안의 32개 이웃
    PointNet → 각 그룹을 128차원 특징으로 압축
    ↓
중간 특징 (npoint=512, feat=128)
    ↓  [Set Abstraction Layer 2]
    FPS → 128개 중심점
    Ball Query → 반경 0.4 안의 64개 이웃
    PointNet → 256차원 특징
    ↓
고수준 특징 (npoint=128, feat=256)
    ↓  [Set Abstraction Layer 3 — Global]
    전체를 하나의 1024차원 벡터로
    ↓
분류 결과 (40클래스)
```

### 핵심 모듈 3가지

| 모듈 | 역할 | 핵심 아이디어 |
|---|---|---|
| **FPS** (Farthest Point Sampling) | 대표 중심점 선택 | 이미 선택된 점들로부터 가장 먼 점을 순서대로 선택 → 균일한 커버리지 |
| **Ball Query** | 이웃 포인트 찾기 | 반경 r 안의 포인트 최대 K개 선택 → 로컬 밀도 적응 |
| **Set Abstraction (SA)** | 지역 특징 추출 | FPS + Ball Query + PointNet을 하나로 묶은 핵심 블록 |

### 왜 FPS인가? (랜덤 샘플링 vs FPS)

- 랜덤 샘플링: 우연히 한쪽에 쏠릴 수 있음
- FPS: 항상 공간을 균등하게 커버 → 더 안정적인 학습

### 왜 Ball Query인가? (KNN vs Ball Query)

- KNN: 밀도에 관계없이 항상 K개 → 밀한 곳과 성긴 곳을 동일하게 취급
- Ball Query: 반경 r 안의 점만 → 로컬 밀도를 자연스럽게 반영

In [ ]:
# Cell 3 — ModelNet40 데이터셋 다운로드 및 로드

# ModelNet40 클래스 이름 (40개)
MODELNET40_CLASSES = [
    'airplane', 'bathtub', 'bed', 'bench', 'bookshelf',
    'bottle', 'bowl', 'car', 'chair', 'cone',
    'cup', 'curtain', 'desk', 'door', 'dresser',
    'flower_pot', 'glass_box', 'guitar', 'keyboard', 'lamp',
    'laptop', 'mantel', 'monitor', 'night_stand', 'person',
    'piano', 'plant', 'radio', 'range_hood', 'sink',
    'sofa', 'stairs', 'stool', 'table', 'tent',
    'toilet', 'tv_stand', 'vase', 'wardrobe', 'xbox'
]
CLASS_KO = [
    '비행기', '욕조', '침대', '벤치', '책장',
    '병', '그릇', '자동차', '의자', '원뿔',
    '컵', '커튼', '책상', '문', '서랍장',
    '화분', '유리상자', '기타', '키보드', '램프',
    '노트북', '벽난로', '모니터', '협탁', '사람',
    '피아노', '화분2', '라디오', '레인지후드', '싱크대',
    '소파', '계단', '스툴', '테이블', '텐트',
    '화장실', 'TV장', '꽃병', '옷장', 'Xbox'
]

NUM_CLASSES = 40
NUM_POINTS = 1024  # 메모리 절약을 위해 2048 → 1024로 서브샘플링


def download_modelnet40(data_dir: Path):
    """ModelNet40 HDF5 데이터셋 다운로드"""
    zip_path = data_dir / 'modelnet40_ply_hdf5_2048.zip'
    extract_dir = data_dir / 'modelnet40_ply_hdf5_2048'

    if extract_dir.exists():
        print("이미 다운로드됨 — 스킵")
        return extract_dir

    url = 'https://shapenet.cs.stanford.edu/media/modelnet40_ply_hdf5_2048.zip'
    print(f"다운로드 중: {url}")
    print("(약 435MB, 시간이 걸릴 수 있습니다...)")

    def _progress(count, block_size, total_size):
        pct = count * block_size / total_size * 100
        if count % 100 == 0:
            print(f"\r  {pct:.1f}%", end='', flush=True)

    urllib.request.urlretrieve(url, zip_path, _progress)
    print("\n압축 해제 중...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_dir)
    zip_path.unlink()  # zip 삭제
    print("완료!")
    return extract_dir


class ModelNet40Dataset(Dataset):
    """ModelNet40 HDF5 포맷 데이터셋"""

    def __init__(self, data_dir: Path, split: str = 'train', num_points: int = 1024,
                 augment: bool = True):
        """
        Args:
            data_dir: modelnet40_ply_hdf5_2048 폴더 경로
            split: 'train' 또는 'test'
            num_points: 사용할 포인트 수 (최대 2048)
            augment: 데이터 증강 여부 (train에서만 적용)
        """
        self.num_points = num_points
        self.augment = augment and (split == 'train')

        # HDF5 파일 목록 읽기
        list_file = data_dir / f'{split}_files.txt'
        with open(list_file, 'r') as f:
            h5_files = [line.strip() for line in f if line.strip()]

        # 경로 처리: 절대경로이면 파일명만 추출해서 data_dir 기준으로 재구성
        self.all_points = []
        self.all_labels = []

        for h5_path in h5_files:
            # 파일명만 추출 (경로 형태가 다양할 수 있음)
            fname = Path(h5_path).name
            full_path = data_dir / fname
            if not full_path.exists():
                full_path = data_dir / h5_path

            with h5py.File(full_path, 'r') as f:
                pts = f['data'][:].astype(np.float32)   # (N, 2048, 3)
                lbl = f['label'][:].flatten().astype(np.int64)  # (N,)
            self.all_points.append(pts)
            self.all_labels.append(lbl)

        self.all_points = np.concatenate(self.all_points, axis=0)  # (전체, 2048, 3)
        self.all_labels = np.concatenate(self.all_labels, axis=0)  # (전체,)
        print(f"[{split}] 샘플 수: {len(self.all_points)}, 클래스 수: {len(np.unique(self.all_labels))}")

    def __len__(self):
        return len(self.all_points)

    def __getitem__(self, idx):
        pts = self.all_points[idx].copy()  # (2048, 3)
        lbl = self.all_labels[idx]

        # 포인트 서브샘플링
        choice = np.random.choice(len(pts), self.num_points, replace=False)
        pts = pts[choice]

        # 정규화: 중심 이동 + 단위구로 스케일
        pts -= pts.mean(axis=0)
        scale = np.max(np.linalg.norm(pts, axis=1))
        if scale > 0:
            pts /= scale

        # 데이터 증강 (train only)
        if self.augment:
            # 랜덤 회전 (Z축)
            theta = np.random.uniform(0, 2 * np.pi)
            rot = np.array([[np.cos(theta), -np.sin(theta), 0],
                            [np.sin(theta),  np.cos(theta), 0],
                            [0,              0,             1]], dtype=np.float32)
            pts = pts @ rot.T
            # 랜덤 지터링
            pts += np.random.normal(0, 0.02, pts.shape).astype(np.float32)
            pts = np.clip(pts, -1.0, 1.0)

        return torch.from_numpy(pts.astype(np.float32)), torch.tensor(lbl, dtype=torch.long)


# ─── 다운로드 및 로드 ───
modelnet_dir = download_modelnet40(DATA_DIR)

train_dataset = ModelNet40Dataset(modelnet_dir, split='train', num_points=NUM_POINTS, augment=True)
test_dataset  = ModelNet40Dataset(modelnet_dir, split='test',  num_points=NUM_POINTS, augment=False)

In [ ]:
# Cell 3b — 샘플 포인트클라우드 시각화 (matplotlib 3D scatter)

def visualize_pointclouds(dataset, n_samples=3, title='ModelNet40 샘플'):
    """포인트클라우드 3D scatter 시각화"""
    # 서로 다른 클래스에서 하나씩 골라 시각화
    seen_labels = set()
    samples = []
    for i in np.random.permutation(len(dataset)):
        pts, lbl = dataset[i]
        label = lbl.item()
        if label not in seen_labels:
            samples.append((pts.numpy(), label))
            seen_labels.add(label)
        if len(samples) == n_samples:
            break

    fig = plt.figure(figsize=(5 * n_samples, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    cmap = plt.cm.viridis
    for i, (pts, label) in enumerate(samples):
        ax = fig.add_subplot(1, n_samples, i + 1, projection='3d')
        # 색상: Z축 높이에 따라 컬러맵 적용
        colors = cmap((pts[:, 2] - pts[:, 2].min()) / (pts[:, 2].ptp() + 1e-8))
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2],
                   c=colors, s=1.5, alpha=0.8)
        ax.set_title(f'{MODELNET40_CLASSES[label]}\n({CLASS_KO[label]})', fontsize=11)
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_box_aspect([1, 1, 1])
        # 축 눈금 제거 (깔끔하게)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])

    plt.tight_layout()
    plt.savefig('sample_pointclouds.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"시각화 저장: sample_pointclouds.png")


visualize_pointclouds(test_dataset, n_samples=3, title='ModelNet40 샘플 포인트클라우드')

# 데이터 분포 확인
label_counts = np.bincount(train_dataset.all_labels, minlength=NUM_CLASSES)
fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(range(NUM_CLASSES), label_counts, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(MODELNET40_CLASSES, rotation=60, ha='right', fontsize=8)
ax.set_ylabel('학습 샘플 수')
ax.set_title('ModelNet40 클래스별 학습 샘플 분포')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"총 학습 샘플: {len(train_dataset):,} | 총 테스트 샘플: {len(test_dataset):,}")

## Cell 4 — 핵심 모듈 직접 구현

아래 3가지 모듈을 **라이브러리 함수 없이** 직접 구현한다.

### ① Farthest Point Sampling (FPS)

```
알고리즘:
1. 첫 번째 중심점: 랜덤으로 하나 선택
2. 현재까지 선택된 점들로부터 모든 점까지의 거리 계산
3. 각 점에서 선택된 점들 중 가장 가까운 점까지의 거리 = 그 점의 "최소 거리"
4. 최소 거리가 가장 큰 점을 다음 중심점으로 선택
5. npoint개가 될 때까지 반복
```

### ② Ball Query

```
알고리즘:
1. 각 중심점(new_xyz)에서 반경 r 안의 점들을 찾기
2. 최대 nsample개 선택
3. r 안에 nsample보다 적은 점이 있으면 → 첫 번째 점으로 패딩
```

### ③ Set Abstraction (SA) Layer

```
입력: (B, N, 3+C) — B=배치, N=포인트 수, C=특징 차원
  ↓ FPS
중심점: (B, npoint, 3)
  ↓ Ball Query
그룹 인덱스: (B, npoint, nsample)
  ↓ 그룹화 + 중심점 상대좌표로 변환
로컬 특징: (B, npoint, nsample, 3+C)
  ↓ Shared MLP (PointNet)
  ↓ Max Pooling (over nsample)
출력: (B, npoint, C')
```

In [ ]:
# Cell 4 — FPS / Ball Query / Set Abstraction 직접 구현

# ─────────────────────────────────────────────
# ① Farthest Point Sampling (FPS)
# ─────────────────────────────────────────────
def farthest_point_sample(xyz: torch.Tensor, npoint: int) -> torch.Tensor:
    """
    가장 먼 점을 순서대로 선택하는 FPS 알고리즘

    Args:
        xyz   : (B, N, 3) — 입력 포인트클라우드
        npoint: 선택할 중심점 수

    Returns:
        centroids: (B, npoint) — 선택된 포인트의 인덱스
    """
    B, N, _ = xyz.shape
    device = xyz.device

    # 결과 인덱스 저장
    centroids = torch.zeros(B, npoint, dtype=torch.long, device=device)

    # 각 포인트의 "현재까지 선택된 점들 중 가장 가까운 거리" 초기값 = 매우 큰 수
    distance = torch.ones(B, N, device=device) * 1e10

    # 첫 번째 중심점: 배치별로 랜덤 선택 (실제로는 0번 고정해도 됨)
    farthest = torch.randint(0, N, (B,), dtype=torch.long, device=device)

    # 배치 인덱스 (반복 사용)
    batch_idx = torch.arange(B, dtype=torch.long, device=device)

    for i in range(npoint):
        # 현재 선택된 점 저장
        centroids[:, i] = farthest

        # 현재 선택된 점의 좌표 추출: (B, 3)
        centroid_xyz = xyz[batch_idx, farthest, :]  # (B, 3)

        # 선택된 점에서 모든 점까지의 거리 계산 (유클리드 거리² — sqrt 생략으로 속도 개선)
        # centroid_xyz: (B, 1, 3) → 브로드캐스팅으로 (B, N) 거리 계산
        dist = torch.sum((xyz - centroid_xyz.unsqueeze(1)) ** 2, dim=-1)  # (B, N)

        # 최소 거리 업데이트: 현재 거리가 기존 최소 거리보다 작으면 업데이트
        distance = torch.minimum(distance, dist)

        # 다음 중심점: 최소 거리가 가장 큰 점 선택 (= 기존 선택점들로부터 가장 먼 점)
        farthest = torch.argmax(distance, dim=-1)  # (B,)

    return centroids


# ─────────────────────────────────────────────
# ② Ball Query
# ─────────────────────────────────────────────
def ball_query(radius: float, nsample: int,
               xyz: torch.Tensor, new_xyz: torch.Tensor) -> torch.Tensor:
    """
    각 중심점(new_xyz) 반경 radius 안의 이웃 포인트 인덱스 반환

    Args:
        radius : 검색 반경
        nsample: 그룹당 최대 포인트 수
        xyz    : (B, N, 3) — 전체 포인트클라우드
        new_xyz: (B, npoint, 3) — 중심점 (FPS 결과)

    Returns:
        group_idx: (B, npoint, nsample) — 그룹별 포인트 인덱스
    """
    B, N, _ = xyz.shape
    _, npoint, _ = new_xyz.shape
    device = xyz.device

    # 기본값: 반경 안에 점이 없을 때 패딩용으로 0번 인덱스 사용
    group_idx = torch.arange(N, dtype=torch.long, device=device)
    group_idx = group_idx.view(1, 1, N).expand(B, npoint, N).clone()  # (B, npoint, N)

    # 각 중심점에서 모든 포인트까지의 거리² 계산
    # new_xyz: (B, npoint, 1, 3), xyz: (B, 1, N, 3) → dist: (B, npoint, N)
    sq_dist = torch.sum(
        (new_xyz.unsqueeze(2) - xyz.unsqueeze(1)) ** 2,
        dim=-1
    )  # (B, npoint, N)

    # 반경 밖의 점은 N(=최대 인덱스)으로 마스킹
    mask = sq_dist > radius ** 2  # True = 반경 밖
    group_idx[mask] = N  # 범위 밖 표시

    # 거리 순으로 정렬해서 가장 가까운 nsample개 선택
    # (N으로 마스킹된 것들은 정렬 시 맨 뒤로 밀림)
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]  # (B, npoint, nsample)

    # 반경 안에 nsample보다 적은 점이 있으면 → 첫 번째 유효 점으로 패딩
    # 첫 번째 인덱스(가장 가까운 점)를 반복해서 채움
    group_first = group_idx[:, :, 0:1].expand_as(group_idx)  # (B, npoint, nsample)
    mask_invalid = group_idx >= N  # 여전히 N인 위치 = 유효 포인트 없음
    group_idx[mask_invalid] = group_first[mask_invalid]

    return group_idx  # (B, npoint, nsample)


# ─────────────────────────────────────────────
# 보조 함수: 인덱스로 포인트 수집
# ─────────────────────────────────────────────
def index_points(points: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    """
    points에서 idx에 해당하는 포인트를 수집

    Args:
        points: (B, N, C)
        idx   : (B, ...) — 임의 형태의 인덱스

    Returns:
        (B, ..., C) — 수집된 포인트
    """
    B = points.shape[0]
    device = points.device
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_idx = torch.arange(B, dtype=torch.long, device=device)
    batch_idx = batch_idx.view(view_shape).repeat(repeat_shape)  # (B, ...)
    return points[batch_idx, idx, :]  # (B, ..., C)


# ─────────────────────────────────────────────
# ③ Set Abstraction (SA) Layer
# ─────────────────────────────────────────────
class SetAbstraction(nn.Module):
    """
    PointNet++의 핵심 블록: Set Abstraction Layer

    FPS → Ball Query → Shared MLP → Max Pooling
    """

    def __init__(self, npoint: int, radius: float, nsample: int,
                 in_channel: int, mlp_channels: list):
        """
        Args:
            npoint      : FPS로 선택할 중심점 수
            radius      : Ball Query 반경
            nsample     : Ball Query 최대 이웃 수
            in_channel  : 입력 특징 차원 (3 포함 — x,y,z + 추가 특징)
            mlp_channels: Shared MLP의 출력 채널 리스트 예: [64, 64, 128]
        """
        super().__init__()
        self.npoint = npoint
        self.radius = radius
        self.nsample = nsample

        # Shared MLP: 1D Conv로 구현 (각 포인트에 동일한 MLP 적용 = weight sharing)
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()

        last_channel = in_channel
        for out_channel in mlp_channels:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel

    def forward(self, xyz: torch.Tensor, points: torch.Tensor = None):
        """
        Args:
            xyz   : (B, N, 3) — 포인트 좌표
            points: (B, N, C) — 포인트 추가 특징 (없으면 None)

        Returns:
            new_xyz   : (B, npoint, 3) — 새 중심점 좌표
            new_points: (B, npoint, mlp[-1]) — 추출된 특징
        """
        # ── Step 1: FPS로 중심점 선택 ──
        fps_idx = farthest_point_sample(xyz, self.npoint)  # (B, npoint)
        new_xyz = index_points(xyz, fps_idx)               # (B, npoint, 3)

        # ── Step 2: Ball Query로 이웃 찾기 ──
        group_idx = ball_query(self.radius, self.nsample, xyz, new_xyz)  # (B, npoint, nsample)
        grouped_xyz = index_points(xyz, group_idx)  # (B, npoint, nsample, 3)

        # 중심점 기준 상대 좌표로 변환 (로컬 구조 표현)
        grouped_xyz -= new_xyz.unsqueeze(2)  # (B, npoint, nsample, 3)

        # 추가 특징 있으면 좌표와 연결
        if points is not None:
            grouped_points = index_points(points, group_idx)  # (B, npoint, nsample, C)
            grouped_points = torch.cat([grouped_xyz, grouped_points], dim=-1)  # (B, npoint, nsample, 3+C)
        else:
            grouped_points = grouped_xyz  # (B, npoint, nsample, 3)

        # ── Step 3: Shared MLP 적용 ──
        # Conv2d 입력 형태로 변환: (B, C, npoint, nsample)
        grouped_points = grouped_points.permute(0, 3, 1, 2).contiguous()

        for conv, bn in zip(self.mlp_convs, self.mlp_bns):
            grouped_points = F.relu(bn(conv(grouped_points)))

        # ── Step 4: Max Pooling (nsample 차원 제거) ──
        # 각 그룹에서 가장 두드러진 특징만 남김
        new_points = grouped_points.max(dim=-1)[0]  # (B, C', npoint)
        new_points = new_points.permute(0, 2, 1).contiguous()  # (B, npoint, C')

        return new_xyz, new_points


# ─── 모듈 단위 테스트 ───
print("모듈 단위 테스트 중...")
B, N, C = 4, 128, 3  # 빠른 테스트용
test_xyz = torch.randn(B, N, C).to(DEVICE)

# FPS 테스트
fps_idx = farthest_point_sample(test_xyz, 32)
print(f"FPS: 입력 {test_xyz.shape} → 중심점 인덱스 {fps_idx.shape}")

# Ball Query 테스트
new_xyz = index_points(test_xyz, fps_idx)
grp_idx = ball_query(0.3, 16, test_xyz, new_xyz)
print(f"Ball Query: 그룹 인덱스 {grp_idx.shape}")

# SA Layer 테스트
sa = SetAbstraction(npoint=32, radius=0.3, nsample=16,
                    in_channel=3, mlp_channels=[32, 32, 64]).to(DEVICE)
out_xyz, out_feat = sa(test_xyz)
print(f"SA Layer: 입력 {test_xyz.shape} → 중심 {out_xyz.shape}, 특징 {out_feat.shape}")
print("\n모든 핵심 모듈 정상 작동!")

In [ ]:
# Cell 5 — PointNet++ 분류 모델 전체 구현

class GlobalSetAbstraction(nn.Module):
    """
    SA Layer 3단계: 전체 포인트를 하나의 벡터로 압축 (Global)
    FPS/Ball Query 없이 모든 포인트에 PointNet 적용 후 Max Pooling
    """

    def __init__(self, in_channel: int, mlp_channels: list):
        super().__init__()
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()

        last_channel = in_channel
        for out_channel in mlp_channels:
            self.mlp_convs.append(nn.Conv1d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm1d(out_channel))
            last_channel = out_channel

    def forward(self, xyz: torch.Tensor, points: torch.Tensor):
        """
        Args:
            xyz   : (B, N, 3)
            points: (B, N, C)

        Returns:
            new_xyz   : None (글로벌 레이어는 좌표 없음)
            new_points: (B, 1, mlp[-1]) — 전체 포인트클라우드의 글로벌 특징
        """
        # 좌표와 특징 연결
        if points is not None:
            combined = torch.cat([xyz, points], dim=-1)  # (B, N, 3+C)
        else:
            combined = xyz  # (B, N, 3)

        # Conv1d 형태로 변환: (B, C, N)
        x = combined.permute(0, 2, 1).contiguous()

        for conv, bn in zip(self.mlp_convs, self.mlp_bns):
            x = F.relu(bn(conv(x)))

        # Global Max Pooling
        x = x.max(dim=-1)[0]  # (B, mlp[-1])
        return None, x


class PointNetPlusPlus(nn.Module):
    """
    PointNet++ 분류 네트워크 (SSG — Single-Scale Grouping)

    구조:
      SA1: npoint=512, r=0.2, nsample=32, mlp=[64, 64, 128]
      SA2: npoint=128, r=0.4, nsample=64, mlp=[128, 128, 256]
      SA3: Global,             mlp=[256, 512, 1024]
      FC : 1024→512→256→num_classes (dropout 0.4)
    """

    def __init__(self, num_classes: int = 40, num_points: int = 1024):
        super().__init__()
        self.num_classes = num_classes

        # Set Abstraction 3단계
        self.sa1 = SetAbstraction(
            npoint=512, radius=0.2, nsample=32,
            in_channel=3,          # 입력: XYZ만
            mlp_channels=[64, 64, 128]
        )
        self.sa2 = SetAbstraction(
            npoint=128, radius=0.4, nsample=64,
            in_channel=3 + 128,    # 입력: XYZ + SA1 출력 특징
            mlp_channels=[128, 128, 256]
        )
        self.sa3 = GlobalSetAbstraction(
            in_channel=3 + 256,    # 입력: XYZ + SA2 출력 특징
            mlp_channels=[256, 512, 1024]
        )

        # 분류 헤드
        self.fc1 = nn.Linear(1024, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.drop2 = nn.Dropout(0.4)

        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, xyz: torch.Tensor):
        """
        Args:
            xyz: (B, N, 3) — 입력 포인트클라우드

        Returns:
            logits: (B, num_classes) — 분류 점수
        """
        B = xyz.shape[0]

        # SA Layer 1: 2048 → 512 포인트
        l1_xyz, l1_pts = self.sa1(xyz, None)          # l1_pts: (B, 512, 128)

        # SA Layer 2: 512 → 128 포인트
        l2_xyz, l2_pts = self.sa2(l1_xyz, l1_pts)    # l2_pts: (B, 128, 256)

        # SA Layer 3 (Global): 128 → 1 벡터
        _, l3_pts = self.sa3(l2_xyz, l2_pts)         # l3_pts: (B, 1024)

        # 분류 헤드
        x = self.drop1(F.relu(self.bn1(self.fc1(l3_pts))))
        x = self.drop2(F.relu(self.bn2(self.fc2(x))))
        x = self.fc3(x)

        return x  # (B, num_classes)

    def count_parameters(self):
        """학습 가능한 파라미터 수 계산"""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ─── 모델 생성 및 확인 ───
model = PointNetPlusPlus(num_classes=NUM_CLASSES, num_points=NUM_POINTS).to(DEVICE)

# 더미 입력으로 forward pass 테스트
dummy = torch.randn(2, NUM_POINTS, 3).to(DEVICE)
with torch.no_grad():
    out = model(dummy)

n_params = model.count_parameters()
print(f"모델 구조 테스트 성공!")
print(f"  입력: {dummy.shape}")
print(f"  출력: {out.shape}")
print(f"  총 파라미터: {n_params:,} ({n_params/1e6:.2f}M)")
print(f"  논문 기준: ~1.7M 파라미터")

In [ ]:
# Cell 6 — 학습 파이프라인

# ─── 하이퍼파라미터 ───
BATCH_SIZE   = 24
NUM_EPOCHS   = 20
LEARNING_RATE = 0.001
WEIGHT_DECAY  = 1e-4

# ─── DataLoader ───
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True, drop_last=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)

print(f"학습 배치 수: {len(train_loader)} | 테스트 배치 수: {len(test_loader)}")

# ─── Optimizer & Scheduler ───
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-5
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # 레이블 스무딩으로 일반화 향상


def train_one_epoch(model, loader, optimizer, criterion, device):
    """1 에포크 학습"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc='학습', leave=False, ncols=100)
    for pts, labels in pbar:
        pts = pts.to(device, non_blocking=True)       # (B, N, 3)
        labels = labels.to(device, non_blocking=True) # (B,)

        optimizer.zero_grad()
        logits = model(pts)                # (B, 40)
        loss = criterion(logits, labels)
        loss.backward()
        # Gradient Clipping (폭발 방지)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # 통계
        total_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)

        pbar.set_postfix({'loss': f'{total_loss/total:.4f}', 'acc': f'{correct/total*100:.1f}%'})

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """검증/테스트 평가"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for pts, labels in loader:
        pts = pts.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(pts)
        loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


# ─── 학습 루프 ───
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
best_acc = 0.0
best_checkpoint = CHECKPOINT_DIR / 'best_model.pth'

print(f"\n학습 시작: {NUM_EPOCHS} 에포크, 배치 {BATCH_SIZE}, lr {LEARNING_RATE}")
print(f"디바이스: {DEVICE}")
print(f"예상 시간: RTX 4080 기준 약 15~20분")
print('=' * 70)

train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    # 학습
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)

    # 평가
    te_loss, te_acc, _, _ = evaluate(model, test_loader, criterion, DEVICE)

    scheduler.step()

    # 기록
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc * 100)
    history['test_loss'].append(te_loss)
    history['test_acc'].append(te_acc * 100)

    # 체크포인트 저장
    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_acc': best_acc,
            'history': history,
        }, best_checkpoint)
        flag = ' ★ 최고!'
    else:
        flag = ''

    elapsed = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr']
    print(f"[{epoch:02d}/{NUM_EPOCHS}] "
          f"train loss={tr_loss:.4f} acc={tr_acc*100:.1f}% | "
          f"test loss={te_loss:.4f} acc={te_acc*100:.1f}% | "
          f"lr={current_lr:.6f} | {elapsed:.0f}s{flag}")

total_time = time.time() - train_start
print('=' * 70)
print(f"학습 완료! 총 시간: {total_time/60:.1f}분")
print(f"최고 Test Accuracy: {best_acc*100:.2f}%")
print(f"체크포인트 저장: {best_checkpoint}")

In [ ]:
# Cell 6b — 학습 곡선 시각화

epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PointNet++ 학습 곡선 (ModelNet40)', fontsize=13, fontweight='bold')

# Loss 그래프
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
ax.plot(epochs_range, history['test_loss'],  'r-o', markersize=4, label='Test Loss')
ax.set_xlabel('에포크')
ax.set_ylabel('Loss')
ax.set_title('Loss 변화')
ax.legend()
ax.grid(alpha=0.3)

# Accuracy 그래프
ax = axes[1]
ax.plot(epochs_range, history['train_acc'], 'b-o', markersize=4, label='Train Acc')
ax.plot(epochs_range, history['test_acc'],  'r-o', markersize=4, label='Test Acc')
ax.axhline(y=88.0, color='gray', linestyle='--', alpha=0.7, label='목표 (88%)')
ax.axhline(y=91.9, color='green', linestyle='--', alpha=0.7, label='논문 (91.9%)')
ax.set_xlabel('에포크')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy 변화')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim([0, 100])

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("학습 곡선 저장: training_curves.png")

In [ ]:
# Cell 7 — 최종 평가 + 시각화

# 최고 체크포인트 로드
checkpoint = torch.load(best_checkpoint, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"체크포인트 로드: epoch {checkpoint['epoch']}, best acc {checkpoint['best_acc']*100:.2f}%")

# 최종 테스트 평가
test_loss, test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, DEVICE)
print(f"\n최종 Test Accuracy: {test_acc*100:.2f}%")
print(f"최종 Test Loss: {test_loss:.4f}")

In [ ]:
# Cell 7b — Confusion Matrix (상위 10개 클래스)

# 클래스별 accuracy 계산
class_correct = np.zeros(NUM_CLASSES)
class_total = np.zeros(NUM_CLASSES)
for pred, label in zip(all_preds, all_labels):
    class_total[label] += 1
    if pred == label:
        class_correct[label] += 1

class_acc = class_correct / (class_total + 1e-8) * 100

# 상위 10개 클래스 선택 (샘플 수 기준)
top10_idx = np.argsort(class_total)[::-1][:10]
top10_labels = all_labels.copy()
top10_preds = all_preds.copy()

# 상위 10개 클래스만 필터링
mask = np.isin(all_labels, top10_idx)
filtered_labels = all_labels[mask]
filtered_preds = all_preds[mask]

# 레이블을 0~9로 재매핑
label_map = {old: new for new, old in enumerate(top10_idx)}
filtered_labels_mapped = np.array([label_map[l] for l in filtered_labels])
filtered_preds_mapped = np.array([label_map.get(p, -1) for p in filtered_preds])
valid_mask = filtered_preds_mapped >= 0
filtered_labels_mapped = filtered_labels_mapped[valid_mask]
filtered_preds_mapped = filtered_preds_mapped[valid_mask]

top10_class_names = [MODELNET40_CLASSES[i] for i in top10_idx]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('PointNet++ 평가 결과', fontsize=13, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(filtered_labels_mapped, filtered_preds_mapped)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=top10_class_names
)
disp.plot(ax=axes[0], colorbar=False, xticks_rotation=45)
axes[0].set_title('Confusion Matrix (상위 10개 클래스)')
axes[0].set_xlabel('예측')
axes[0].set_ylabel('실제')

# 클래스별 Accuracy 막대 그래프
sorted_idx = np.argsort(class_acc)[::-1]
ax = axes[1]
colors = ['#2ecc71' if acc >= 88 else '#e74c3c' if acc < 70 else '#f39c12'
          for acc in class_acc[sorted_idx]]
bars = ax.bar(range(NUM_CLASSES), class_acc[sorted_idx], color=colors, edgecolor='none')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels([MODELNET40_CLASSES[i] for i in sorted_idx],
                    rotation=60, ha='right', fontsize=7)
ax.axhline(y=88, color='gray', linestyle='--', alpha=0.7, label='목표 (88%)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('클래스별 Accuracy (정확도 순 정렬)')
ax.set_ylim([0, 105])
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 범례 추가
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='≥88% (목표 달성)'),
    Patch(facecolor='#f39c12', label='70~88%'),
    Patch(facecolor='#e74c3c', label='<70% (개선 필요)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("평가 결과 저장: evaluation_results.png")

# 상위/하위 5개 클래스 출력
print(f"\n클래스별 Accuracy 상위 5개:")
for rank, idx in enumerate(sorted_idx[:5], 1):
    print(f"  {rank}. {MODELNET40_CLASSES[idx]:15s} ({CLASS_KO[idx]}) — {class_acc[idx]:.1f}%")

print(f"\n클래스별 Accuracy 하위 5개:")
for rank, idx in enumerate(sorted_idx[-5:][::-1], 1):
    print(f"  {rank}. {MODELNET40_CLASSES[idx]:15s} ({CLASS_KO[idx]}) — {class_acc[idx]:.1f}%")

In [ ]:
# Cell 7c — 틀린 예측 시각화 (포인트클라우드 scatter 3D)

# 틀린 샘플 찾기
wrong_mask = all_preds != all_labels
wrong_indices_in_test = np.where(wrong_mask)[0]

print(f"총 테스트 샘플: {len(all_labels):,}")
print(f"틀린 샘플: {wrong_mask.sum():,} ({wrong_mask.mean()*100:.1f}%)")

# 틀린 샘플 중 흥미로운 것 선택 (다양한 클래스에서)
n_show = 4
seen_wrong_classes = set()
wrong_examples = []

for idx in wrong_indices_in_test:
    true_label = all_labels[idx]
    pred_label = all_preds[idx]
    if true_label not in seen_wrong_classes:
        pts, _ = test_dataset[idx]
        wrong_examples.append((pts.numpy(), true_label, pred_label))
        seen_wrong_classes.add(true_label)
    if len(wrong_examples) == n_show:
        break

fig = plt.figure(figsize=(5 * n_show, 5))
fig.suptitle('틀린 예측 예시 (포인트클라우드)', fontsize=13, fontweight='bold')

for i, (pts, true_lbl, pred_lbl) in enumerate(wrong_examples):
    ax = fig.add_subplot(1, n_show, i + 1, projection='3d')
    # 색상: 높이에 따른 컬러맵
    colors = plt.cm.RdYlBu((pts[:, 2] - pts[:, 2].min()) / (pts[:, 2].ptp() + 1e-8))
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=colors, s=2, alpha=0.8)

    true_name = MODELNET40_CLASSES[true_lbl]
    pred_name = MODELNET40_CLASSES[pred_lbl]
    ax.set_title(
        f"정답: {true_name}\n예측: {pred_name}",
        fontsize=9,
        color='red' if true_name != pred_name else 'green'
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

plt.tight_layout()
plt.savefig('wrong_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print("틀린 예측 시각화 저장: wrong_predictions.png")

## Cell 8 — 논문 vs 구현 결과 비교

| 항목 | 논문 (Qi et al. 2017) | 우리 구현 |
|---|---|---|
| **Test Accuracy** | 91.9% | 실행 후 채울 것 |
| **파라미터 수** | ~1.7M | ~1.4M (1024포인트 기준) |
| **입력 포인트 수** | 1024 | 1024 |
| **학습 에포크** | 200 | 20 |
| **핵심 기여 — FPS** | ✅ 직접 구현 | ✅ 직접 구현 |
| **핵심 기여 — Ball Query** | ✅ 직접 구현 | ✅ 직접 구현 |
| **핵심 기여 — Set Abstraction** | ✅ 직접 구현 | ✅ 직접 구현 |
| **데이터 증강** | 랜덤 회전 + 지터링 | 랜덤 회전 + 지터링 ✅ |
| **옵티마이저** | Adam | Adam ✅ |

### 성능 차이 원인 분석

1. **에포크 차이**: 논문은 200 에포크 학습 → 우리는 20 에포크 (포트폴리오 시연 목적)
2. **Vote 앙상블 없음**: 논문은 테스트 시 랜덤 변환 10회 평균 → 우리는 단순 1회 추론
3. **Multi-Scale Grouping 없음**: 논문의 MSG 변형은 더 높은 성능이지만 구현 복잡도 증가
4. **배치 크기**: 논문은 32 → 우리는 24 (메모리 제약)

### 핵심 기여 달성 여부

이 구현의 목적은 **원리를 직접 코드로 구현**하는 것이다.

- FPS 알고리즘: PyTorch 텐서 연산으로 직접 구현 ✅
- Ball Query: 거리 기반 이웃 검색 직접 구현 ✅  
- Set Abstraction: 3단계 계층적 학습 구조 직접 구현 ✅
- 평가 파이프라인: Confusion Matrix + 클래스별 Accuracy 시각화 ✅

### 개선 방향

1. **속도**: FPS/Ball Query를 CUDA 커널로 직접 구현하면 10배 이상 빠름 (pointnet2_ops 라이브러리 참조)
2. **성능**: Multi-Scale Grouping (MSG), Vote Ensemble 추가
3. **확장**: Segmentation 헤드 추가 (Feature Propagation Layer)

## Cell 9 — 자율주행 연결: PointNet++이 어떻게 쓰이는가

### LiDAR 포인트클라우드 처리 계보

```
LiDAR 센서
    ↓
포인트클라우드 (수만 ~ 수십만 포인트)
    ↓
PointNet++ (2017) — 계층적 로컬 특징 학습의 시작
    ↓
VoxelNet (2018) — 공간을 Voxel로 분할 후 특징 추출 (PointNet 활용)
    ↓
PointPillars (2019) — Voxel → Pillar (수직 기둥), 훨씬 빠름
    ↓
CenterPoint (2021) — Center-based 3D 탐지 (실시간 가능)
    ↓
현재 자율주행 인식 스택 (Tesla FSD, Waymo, Cruise 등)
```

### KITTI 데이터셋에서의 적용

자율주행의 표준 벤치마크인 KITTI LiDAR 데이터:

```python
# KITTI 포인트클라우드 예시
# shape: (N, 4) — x, y, z, intensity
# N ≈ 120,000 포인트 (Velodyne HDL-64E 기준)

# ModelNet40 vs KITTI 차이점:
# ModelNet40: 정규화된 단일 물체 (자동차 1개)
# KITTI: 실제 도로 장면 전체 (자동차 수십 대 + 보행자 + 건물 + 도로...)

# PointNet++ → KITTI 3D 검출 파이프라인:
# 1. 바닥 제거 (RANSAC 또는 높이 필터)
# 2. FPS로 관심 영역 샘플링
# 3. SA Layer로 3D 특징 추출
# 4. Region Proposal → 3D Bounding Box 예측
#    (x, y, z, w, h, l, heading)
```

### 핵심 차이점: 실제 자율주행 환경

| 항목 | ModelNet40 (이번 구현) | KITTI (실제 자율주행) |
|---|---|---|
| 포인트 수 | 1,024 | ~120,000 |
| 장면 구성 | 단일 물체 | 전체 3D 장면 |
| 태스크 | 분류 (40클래스) | 3D 검출 (위치+크기+방향) |
| 실시간성 | 불필요 | 10 Hz 이상 필수 |
| 밀도 | 균일 | 거리에 따라 급격히 변화 |

### PointNet++의 자율주행 기여

1. **보행자 탐지**: 포인트 수가 적어도 (10~50개) 형태를 인식
2. **원거리 물체**: 성긴 포인트에서도 Ball Query로 로컬 구조 유지
3. **센서 융합**: LiDAR 특징 + 카메라 특징을 SA Layer 수준에서 융합 (BEVFusion)

---

### 다음 노트북 예고: `02_kitti_3d_detection.ipynb`

```
02_kitti_3d_detection.ipynb
├── KITTI 데이터셋 로드 (bin 파일 → numpy)
├── 포인트클라우드 전처리 (바닥 제거, ROI 필터링)
├── PointPillars 구조 이해 및 핵심 모듈 구현
├── 3D IoU 기반 평가 파이프라인
└── BEV 시각화 (새 위에서 본 자동차 탐지 결과)
```

**PointNet++ → PointPillars로 이어지는 개념적 연결:**
- FPS → Pillar 생성 (고정 격자 기반으로 단순화)
- Set Abstraction → Pillar Feature Net (각 Pillar 내 PointNet)
- Hierarchical SA → 2D Pseudo-image → 2D CNN Backbone